In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "fans-t4-qa"
TOWER = 4
TOWER_NAME = "Tower of Fans"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 4: Tower of Fans


In [2]:
# F1 -- Community Engagement & Feedback

# Probe: feedback references in HTML pages (start.html has feedback button)
start_html = read_file(WEB / "start.html")
has_feedback_html = bool(re.search(r"feedback|send.*feedback", start_html, re.IGNORECASE))
record("F1-P1", "Feedback mechanism in start page", has_feedback_html)

# Probe: feedback styles in CSS
site_css = read_file(WEB / "css" / "site.css")
has_feedback_css = bool(re.search(r"feedback|modal", site_css, re.IGNORECASE))
record("F1-P2", "Feedback UI styles exist", has_feedback_css)

# Probe: support/contact references on pages
contact_refs = 0
for hf in list(WEB.glob("*.html")):
    content = read_file(hf)
    if re.search(r"support|contact|help|feedback", content, re.IGNORECASE):
        contact_refs += 1
record("F1-P3", "Support/contact references across pages", contact_refs >= 2, f"{contact_refs} pages")

# Probe: home page has engagement elements
home_html = read_file(WEB / "home.html")
has_engagement = bool(re.search(r"community|contribute|join|feedback|share|app", home_html, re.IGNORECASE))
record("F1-P4", "Home page has community engagement", has_engagement)

PASS: Feedback mechanism in start page
PASS: Feedback UI styles exist
PASS: Support/contact references across pages -- 12 pages
PASS: Home page has community engagement


In [3]:
# F2 -- Open Source & Contributing

has_license = (BROWSER_ROOT / "LICENSE").exists()
record("F2-P1", "LICENSE file exists", has_license)

readme = read_file(BROWSER_ROOT / "README.md")
record("F2-P2", "README.md exists", len(readme) > 200, f"{len(readme)} bytes")

pkg = read_file(BROWSER_ROOT / "package.json")
has_meta = bool(re.search(r'"name"|"version"', pkg))
record("F2-P3", "package.json has metadata", has_meta)

wf_dir = BROWSER_ROOT / ".github" / "workflows"
workflows = list(wf_dir.glob("*.yml")) if wf_dir.exists() else []
record("F2-P4", "GitHub Actions CI exists", len(workflows) >= 1, f"{len(workflows)} workflows")

PASS: LICENSE file exists
PASS: README.md exists -- 4055 bytes
PASS: package.json has metadata
PASS: GitHub Actions CI exists -- 4 workflows


In [4]:
# F3 -- Brand & Identity

logo_dir = WEB / "images" / "yinyang"
logos = list(logo_dir.glob("yinyang-logo-*.png")) if logo_dir.exists() else []
record("F3-P1", "Multiple logo sizes", len(logos) >= 3, f"{len(logos)} logos")

has_svg = (WEB / "favicon.svg").exists()
has_ico = (WEB / "favicon.ico").exists()
record("F3-P2", "Favicon files exist", has_svg or has_ico)

manifest = read_file(WEB / "manifest.json")
has_brand = bool(re.search(r'"name"|"theme_color"', manifest))
record("F3-P3", "PWA manifest has branding", has_brand)

site_css = read_file(WEB / "css" / "site.css")
brand_vars = len(re.findall(r"--sb-(?:accent|signal|bg|text|border)", site_css))
record("F3-P4", "Brand CSS variables defined", brand_vars >= 5, f"{brand_vars} vars")

PASS: Multiple logo sizes -- 7 logos
PASS: Favicon files exist
PASS: PWA manifest has branding
PASS: Brand CSS variables defined -- 658 vars


In [5]:
# F4 -- Social Sharing & Discovery

og_count = 0
for hf in list(WEB.glob("*.html")):
    content = read_file(hf)
    if re.search(r"og:title|og:description", content):
        og_count += 1
record("F4-P1", "Pages have OG meta tags", og_count >= 5, f"{og_count} pages")

tw_count = 0
for hf in list(WEB.glob("*.html")):
    content = read_file(hf)
    if re.search(r"twitter:card|twitter:title", content):
        tw_count += 1
record("F4-P2", "Pages have Twitter cards", tw_count >= 3, f"{tw_count} pages")

download = read_file(WEB / "download.html")
record("F4-P3", "Download page exists", len(download) > 200)

PASS: Pages have OG meta tags -- 13 pages
PASS: Pages have Twitter cards -- 11 pages
PASS: Download page exists


In [6]:
# F5 -- Delight & Trust

delight = read_file(WEB / "js" / "yinyang-delight.js")
record("F5-P1", "Delight engine JS exists", len(delight) > 100, f"{len(delight)} bytes")

theme_js = read_file(WEB / "js" / "theme.js")
has_themes = bool(re.search(r"theme|dark|light", theme_js, re.IGNORECASE))
record("F5-P2", "Theme system for personalization", has_themes)

locale_dir = BROWSER_ROOT / "app" / "locales" / "yinyang"
locale_count = len(list(locale_dir.glob("*.json"))) if locale_dir.exists() else 0
record("F5-P3", "Multi-language support", locale_count >= 10, f"{locale_count} locales")

demo = read_file(WEB / "demo.html")
record("F5-P4", "Demo page exists", len(demo) > 200)

has_security = (BROWSER_ROOT / "SECURITY.md").exists()
record("F6-P1", "SECURITY.md exists", has_security)

csp_count = 0
for hf in list(WEB.glob("*.html")):
    content = read_file(hf)
    if "Content-Security-Policy" in content:
        csp_count += 1
record("F6-P2", "CSP meta tags on pages", csp_count >= 5, f"{csp_count} pages")

js_files = list((WEB / "js").glob("*.js"))
eval_count = 0
for jf in js_files:
    content = read_file(jf)
    eval_count += len(re.findall(r'\beval\s*\(', content))
record("F6-P3", "No eval() in JS", eval_count == 0, f"{eval_count} eval calls")

chain_py = read_file(SRC / "audit" / "chain.py")
has_chain = bool(re.search(r"sha256|hash|seal|verify", chain_py, re.IGNORECASE))
record("F6-P4", "Evidence chain for accountability", has_chain)

PASS: Delight engine JS exists -- 14559 bytes
PASS: Theme system for personalization
PASS: Multi-language support -- 47 locales
PASS: Demo page exists
PASS: SECURITY.md exists
PASS: CSP meta tags on pages -- 17 pages
PASS: No eval() in JS -- 0 eval calls
PASS: Evidence chain for accountability


In [7]:
# EVIDENCE SUMMARY
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
conv = "YES" if summary["converged"] else "NO"
ehash = summary['evidence_hash']
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {conv}")
print(f"Evidence hash: {ehash}")
if summary["findings"]:
    print()
    print("FINDINGS:")
    for fi in summary["findings"]:
        fid = fi['id']
        fname = fi['name']
        fdetail = fi.get('detail', '')
        print(f"  [{fid}] {fname}: {fdetail}")
print()
print(json.dumps(summary, indent=2))


TOWER 4: Tower of Fans
Probes: 23 | Passed: 23 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: e87a02817732e615

{
  "surface": "fans-t4-qa",
  "tower": 4,
  "tower_name": "Tower of Fans",
  "timestamp": "2026-03-06T11:28:29.106185",
  "total_probes": 23,
  "passed": 23,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "e87a02817732e615"
}
